# Clothing Classifier — Training

Transfer learning with `timm` (ResNet-18). Runs on CPU for local dev and moves
to GPU unchanged (the device is auto-detected). 15 clothing classes, 500 images each.

## 1. Imports

In [16]:
import os
from collections import Counter

import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

import timm
from timm.data import resolve_data_config, create_transform

## 2. Configuration
Every tunable knob lives here — nothing to hunt for in the cells below.

In [2]:
# --- data ---
DATA_DIR   = r"C:\Users\daru1\OneDrive\Desktop\Portfolio\Clothes_Dataset"
CKPT_DIR   = "checkpoints"

# --- model ---
MODEL_NAME = "resnet18"      # bump to "resnet50" / "convnext_tiny" on GPU

# --- training ---
BATCH_SIZE      = 32
VAL_SPLIT       = 0.2
SEED            = 42
FREEZE_BACKBONE = True       # True = train only the head (fast on CPU)
EPOCHS          = 3          # 3 for the frozen CPU smoke-test; 15-30 on GPU
LR              = 1e-3 if FREEZE_BACKBONE else 1e-4
WEIGHT_DECAY    = 1e-4

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## 3. Dataset & class inspection
Confirm the folder layout: one subfolder per class, balanced counts.

In [3]:
base = datasets.ImageFolder(DATA_DIR)
print("classes found:", len(base.classes))
print("total images :", len(base))
print()
counts = Counter(base.targets)
for i, cls in enumerate(base.classes):
    print(f"{cls:<25} {counts[i]}")

classes found: 15
total images : 7500

Blazer                    500
Celana_Panjang            500
Celana_Pendek             500
Gaun                      500
Hoodie                    500
Jaket                     500
Jaket_Denim               500
Jaket_Olahraga            500
Jeans                     500
Kaos                      500
Kemeja                    500
Mantel                    500
Polo                      500
Rok                       500
Sweter                    500


## 4. Model & transforms
Transforms are pulled from the model so normalization always matches the backbone.

In [4]:
num_classes = len(base.classes)
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=num_classes)

cfg = resolve_data_config({}, model=model)
train_tf = create_transform(**cfg, is_training=True)    # with augmentation
val_tf   = create_transform(**cfg, is_training=False)   # clean, for eval

model = model.to(device)
print("input config:", cfg)

input config: {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.95, 'crop_mode': 'center'}


## 5. Train / validation split & DataLoaders
Stratified 80/20 split — keeps 100 of each class in validation.

In [5]:
train_idx, val_idx = train_test_split(
    range(len(base.targets)),
    test_size=VAL_SPLIT,
    stratify=base.targets,
    random_state=SEED,
)

train_ds = Subset(datasets.ImageFolder(DATA_DIR, transform=train_tf), train_idx)
val_ds   = Subset(datasets.ImageFolder(DATA_DIR, transform=val_tf),   val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("train:", len(train_ds), " val:", len(val_ds))   # expect 6000 / 1500

train: 6000  val: 1500


## 6. Sanity-check one batch
Catch any image-reading problem in seconds, before a full epoch.

In [6]:
imgs, labels = next(iter(train_loader))
print("batch images:", imgs.shape)   # expect [32, 3, 224, 224]
print("batch labels:", labels[:8])

batch images: torch.Size([32, 3, 224, 224])
batch labels: tensor([11, 11,  4, 10,  4, 14, 14,  0])


## 7. Training setup
Freeze policy, loss, optimizer.

In [7]:
# freeze everything, then re-enable just the classifier head
if FREEZE_BACKBONE:
    for p in model.parameters():
        p.requires_grad = False
    for p in model.get_classifier().parameters():
        p.requires_grad = True

criterion = nn.CrossEntropyLoss()
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

print(f"trainable params: {sum(p.numel() for p in params):,}")

trainable params: 7,695


## 8. Training loop

In [8]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, correct, seen = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            seen += imgs.size(0)
    return total_loss / seen, correct / seen

In [9]:
best_acc = 0.0
os.makedirs(CKPT_DIR, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    print(f"epoch {epoch}/{EPOCHS} | "
          f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}")

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({
            "model_state": model.state_dict(),
            "model_name": MODEL_NAME,
            "classes": base.classes,
            "class_to_idx": base.class_to_idx,
            "input_size": cfg["input_size"],
        }, os.path.join(CKPT_DIR, "best.pt"))
        print(f"  saved best (val acc {best_acc:.3f})")

print("done. best val acc:", round(best_acc, 3))

epoch 1/3 | train loss 2.489 acc 0.237 | val loss 2.233 acc 0.362
  saved best (val acc 0.362)


epoch 2/3 | train loss 2.176 acc 0.380 | val loss 1.974 acc 0.457
  saved best (val acc 0.457)


epoch 3/3 | train loss 2.012 acc 0.404 | val loss 1.814 acc 0.475
  saved best (val acc 0.475)
done. best val acc: 0.475


## 9. Evaluation
_Next: confusion matrix + per-class report on `best.pt`._

In [10]:
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [11]:
UNFREEZE_EPOCHS = 5
FT_LR = 1e-4   

In [12]:
for p in model.parameters():
    p.requires_grad = True

In [13]:
# fresh optimizer over ALL params (the old one only knew about the head)
optimizer = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=1e-4)

# cosine schedule: LR eases down toward 0 over the run for a cleaner finish
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=UNFREEZE_EPOCHS)

for epoch in range(1, UNFREEZE_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"[finetune] epoch {epoch}/{UNFREEZE_EPOCHS} | "
          f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}")

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({
            "model_state": model.state_dict(),
            "model_name": MODEL_NAME,
            "classes": base.classes,
            "class_to_idx": base.class_to_idx,
            "input_size": cfg["input_size"],
        }, "checkpoints/best.pt")
        print(f"  saved best (val acc {best_acc:.3f})")

print("finetune done. best val acc:", round(best_acc, 3))

[finetune] epoch 1/5 | train loss 1.783 acc 0.455 | val loss 1.448 acc 0.549
  saved best (val acc 0.549)


[finetune] epoch 2/5 | train loss 1.597 acc 0.499 | val loss 1.316 acc 0.581
  saved best (val acc 0.581)


[finetune] epoch 3/5 | train loss 1.483 acc 0.534 | val loss 1.232 acc 0.610
  saved best (val acc 0.610)


[finetune] epoch 4/5 | train loss 1.445 acc 0.538 | val loss 1.199 acc 0.622
  saved best (val acc 0.622)


[finetune] epoch 5/5 | train loss 1.399 acc 0.557 | val loss 1.195 acc 0.626
  saved best (val acc 0.626)
finetune done. best val acc: 0.626


In [14]:
# --- Phase 3: keep fine-tuning, longer schedule ---
MORE_EPOCHS = 15
FT_LR = 1e-4

# model is already fully unfrozen from Phase 2; just give it a fresh optimizer + schedule
optimizer = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MORE_EPOCHS)

for epoch in range(1, MORE_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"[phase3] epoch {epoch}/{MORE_EPOCHS} | "
          f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}")

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({
            "model_state": model.state_dict(),
            "model_name": MODEL_NAME,
            "classes": base.classes,
            "class_to_idx": base.class_to_idx,
            "input_size": cfg["input_size"],
        }, "checkpoints/best.pt")
        print(f"  saved best (val acc {best_acc:.3f})")

print("phase3 done. best val acc:", round(best_acc, 3))

[phase3] epoch 1/15 | train loss 1.388 acc 0.557 | val loss 1.127 acc 0.637
  saved best (val acc 0.637)


[phase3] epoch 2/15 | train loss 1.311 acc 0.579 | val loss 1.075 acc 0.657
  saved best (val acc 0.657)


[phase3] epoch 3/15 | train loss 1.272 acc 0.590 | val loss 1.019 acc 0.672
  saved best (val acc 0.672)


[phase3] epoch 4/15 | train loss 1.229 acc 0.599 | val loss 0.986 acc 0.672


[phase3] epoch 5/15 | train loss 1.169 acc 0.623 | val loss 0.971 acc 0.680
  saved best (val acc 0.680)


...

[phase3] epoch 11/15 | train loss 1.033 acc 0.658 | val loss 0.877 acc 0.697


[phase3] epoch 12/15 | train loss 1.020 acc 0.665 | val loss 0.884 acc 0.707
  saved best (val acc 0.707)


[phase3] epoch 13/15 | train loss 1.023 acc 0.671 | val loss 0.881 acc 0.705


[phase3] epoch 14/15 | train loss 1.030 acc 0.663 | val loss 0.869 acc 0.705


[phase3] epoch 15/15 | train loss 1.016 acc 0.670 | val loss 0.879 acc 0.704
phase3 done. best val acc: 0.707


In [15]:
print(model)   # full layer tree

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (drop_block): Identity()
      (act1): ReLU(inplace=True)
      (aa): Identity()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (act2): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (b

In [17]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        out = model(imgs.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=base.classes))
print(confusion_matrix(all_labels, all_preds))

                precision    recall  f1-score   support

        Blazer       0.64      0.54      0.59       100
Celana_Panjang       0.74      0.72      0.73       100
 Celana_Pendek       0.82      0.75      0.78       100
          Gaun       0.63      0.82      0.71       100
        Hoodie       0.79      0.76      0.78       100
         Jaket       0.58      0.61      0.59       100
   Jaket_Denim       0.85      0.82      0.83       100
Jaket_Olahraga       0.66      0.56      0.61       100
         Jeans       0.78      0.84      0.81       100
          Kaos       0.72      0.86      0.79       100
        Kemeja       0.65      0.68      0.66       100
        Mantel       0.66      0.72      0.69       100
          Polo       0.63      0.50      0.56       100
           Rok       0.70      0.62      0.66       100
        Sweter       0.74      0.76      0.75       100

      accuracy                           0.70      1500
     macro avg       0.70      0.70      0.70 